In [1]:
import os
from datasets import load_dataset
import pandas as pd

print("Chargement du dataset CATMuS (depuis le cache)...")
dataset = load_dataset("CATMuS/medieval")

print("Reconstruction du tableau avec index HF préservé...")
df = dataset['train'].remove_columns(['im']).to_pandas()

# Exactement le même filtre que dans 01_exploration_catmus.ipynb (25812 lignes attendues)
df_filtered = df[
    (df['century'].isin(['14', '15', 14, 15])) &
    (df['language'].isin(['French']))
].copy()

print(f"Lignes conservées : {len(df_filtered)}")

# ÉTAPE CLÉ : on capture le lien vers l'image AVANT tout reset_index
df_filtered['source_idx'] = df_filtered.index
df_filtered['image_path'] = df_filtered['source_idx'].apply(lambda idx: f"images/ligne_{idx}.png")
df_filtered['source_corpus'] = 'CATMuS'

# Vérification : combien d'images existent vraiment au bon endroit ?
base_dir = "../dataset_nlp"
existe = df_filtered['image_path'].apply(lambda p: os.path.exists(os.path.join(base_dir, p)))
print(f"Images trouvées : {existe.sum()} / {len(df_filtered)}")
if existe.sum() < len(df_filtered):
    print("Exemples de lignes sans image :")
    print(df_filtered[~existe][['source_idx','century','shelfmark','project']].head(20))

df_filtered = df_filtered.reset_index(drop=True)
df_filtered.to_csv("../dataset_nlp/catmus_french_14_15_clean.csv", index=False)
print("✅ catmus_french_14_15_clean.csv généré.")

/root/htr-medieval-manuscripts-XIVe/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement du dataset CATMuS (depuis le cache)...


Reconstruction du tableau avec index HF préservé...
Lignes conservées : 25812
Images trouvées : 25812 / 25812
✅ catmus_french_14_15_clean.csv généré.


In [2]:
fr22549 = df_filtered[df_filtered['shelfmark'] == 'Paris, BnF, fr. 22549']
print(f"Lignes BnF fr.22549 déjà dans CATMuS (avec image correcte) : {len(fr22549)}")
print(fr22549['project'].value_counts())

Lignes BnF fr.22549 déjà dans CATMuS (avec image correcte) : 2615
project
CREMMA    2615
Name: count, dtype: int64
